In [ ]:
# JOUR 3 — MODÉLISATION
# ============================================
# 1. IMPORTS
# ============================================
import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler


In [ ]:
# ============================================
# 2. CHARGEMENT DES DONNÉES
# ============================================
df = pd.read_csv("../House_Prices.csv")

print("Shape :", df.shape)
print("SalePrice manquantes :", df["SalePrice"].isna().sum())
print("SalePrice = 0 :", (df["SalePrice"] == 0).sum())


In [ ]:
# ============================================
# 3. PRÉPARATION REPRODUCTIBLE DES DONNÉES
# ============================================
model_df = df.copy()
print("Shape des données de modélisation :", model_df.shape)


In [ ]:
# ============================================
# 4. FEATURE ENGINEERING REPRODUCTIBLE
# ============================================
model_df["TotalSF"] = (
    model_df["TotalBsmtSF"].fillna(0)
    + model_df["1stFlrSF"].fillna(0)
    + model_df["2ndFlrSF"].fillna(0)
)
model_df["TotalBathrooms"] = (
    model_df["FullBath"].fillna(0)
    + 0.5 * model_df["HalfBath"].fillna(0)
    + model_df["BsmtFullBath"].fillna(0)
    + 0.5 * model_df["BsmtHalfBath"].fillna(0)
)
model_df["HouseAge"] = model_df["YrSold"] - model_df["YearBuilt"]
model_df["RemodAge"] = model_df["YrSold"] - model_df["YearRemodAdd"]

engineered_features = ["TotalSF", "TotalBathrooms", "HouseAge", "RemodAge"]
print(model_df[engineered_features].head())


In [ ]:
# ============================================
# 5. SÉPARATION DE X ET y
# ============================================
y = model_df["SalePrice"]
X = model_df.drop(columns=["SalePrice"])
X = X.drop(columns=["Id"])

print("Shape de X :", X.shape)
print("Shape de y :", y.shape)
print("Features créées dans X :", set(engineered_features).issubset(X.columns))


In [ ]:
# ============================================
# 6. TRAIN / TEST SPLIT
# ============================================
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

print("X_train :", X_train.shape)
print("X_test  :", X_test.shape)


In [ ]:
# ============================================
# 7. TYPES DE VARIABLES
# ============================================
numeric_features = X_train.select_dtypes(include=["int64", "float64"]).columns
categorical_features = X_train.select_dtypes(include=["object"]).columns

print("Variables numériques :", len(numeric_features))
print("Variables catégorielles :", len(categorical_features))


In [ ]:
# ============================================
# 8. PREPROCESSING
# ============================================
numeric_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ]
)

categorical_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features)
    ]
)


In [31]:
# ============================================
# 9. MODÈLES DU JOUR 3
# ============================================
models = {
    "Linear Regression": LinearRegression(),
    "Random Forest": RandomForestRegressor(
        n_estimators=200, random_state=42, n_jobs=-1
    ),
    "Gradient Boosting": GradientBoostingRegressor(
        n_estimators=200, learning_rate=0.05, max_depth=3, random_state=42
    ),
}


In [ ]:
# ============================================
# 10. ENTRAÎNEMENT, PRÉDICTIONS ET ÉVALUATION
# ============================================
results = []
fitted_models = {}

for name, model in models.items():
    pipeline = Pipeline(
        steps=[("preprocessor", preprocessor), ("model", model)]
    )
    pipeline.fit(X_train, y_train)
    predictions = pipeline.predict(X_test)

    results.append({
        "Model": name,
        "MAE": mean_absolute_error(y_test, predictions),
        "RMSE": np.sqrt(mean_squared_error(y_test, predictions)),
        "R2": r2_score(y_test, predictions),
    })
    fitted_models[name] = pipeline


In [ ]:
# ============================================
# 11. TABLEAU COMPARATIF
# ============================================
comparison = pd.DataFrame(results).sort_values("RMSE").reset_index(drop=True)
comparison


# 12. Analyse des modèles

Le tableau comparatif doit être lu ainsi :

- une **MAE** faible signifie que l’erreur moyenne de prédiction est faible ;
- une **RMSE** faible signifie que les erreurs importantes sont limitées ;
- un **R²** élevé signifie que le modèle explique une grande partie de la variation de `SalePrice`.

Le meilleur modèle initial est celui qui a la **RMSE la plus faible**, en vérifiant aussi la MAE et le R².

## Linear Regression

**Avantages :**
- modèle simple et rapide ;
- relativement facile à interpréter ;
- bon modèle de référence.

**Limites :**
- représente moins bien les relations complexes ;
- peut être moins performante si les variables interagissent fortement.

## Random Forest

**Avantages :**
- capture les relations non linéaires ;
- prend en compte les interactions entre variables ;
- généralement robuste.

**Limites :**
- moins interprétable qu’une régression linéaire ;
- plus coûteux à entraîner ;
- peut moins bien extrapoler sur de nouveaux cas atypiques.

## Gradient Boosting

**Avantages :**
- corrige progressivement les erreurs des arbres précédents ;
- souvent performant sur les données tabulaires ;
- modélise des relations complexes.

**Limites :**
- sensible aux hyperparamètres ;
- risque de surapprentissage ;
- nécessite une optimisation qui sera réalisée au Jour 4 avec la validation croisée.

## Choix du modèle

Le choix final reposera d’abord sur la **RMSE**, puis sera confirmé avec la **MAE** et le **R²**. Il faudra aussi considérer le compromis entre précision et interprétabilité.

# Jour 4 — validation, optimisation et analyse des erreurs

On garde le jeu de test de côté jusqu’à la fin. La validation croisée et la recherche des paramètres utilisent uniquement les données d’entraînement.


In [ ]:
from sklearn.model_selection import KFold, cross_validate, GridSearchCV
import matplotlib.pyplot as plt


## 1. Validation croisée

On partage les données d'entraînement en 5 parties (folds). À chaque tour, le modèle apprend sur 4 parties et est évalué sur la cinquième. La moyenne donne une idée plus stable de ses performances.


In [ ]:
# Même découpage à chaque exécution pour pouvoir comparer les résultats.
folds = KFold(n_splits=5, shuffle=True, random_state=42)

scores_cv = []
for nom, pipeline in fitted_models.items():
    scores = cross_validate(
        pipeline,
        X_train,
        y_train,
        cv=folds,
        scoring={
            "mae": "neg_mean_absolute_error",
            "rmse": "neg_root_mean_squared_error",
            "r2": "r2",
        },
        n_jobs=-1,
    )
    scores_cv.append({
        "Modèle": nom,
        "MAE moyenne": -scores["test_mae"].mean(),
        "RMSE moyenne": -scores["test_rmse"].mean(),
        "R² moyen": scores["test_r2"].mean(),
    })

tableau_cv = pd.DataFrame(scores_cv).sort_values("RMSE moyenne")
tableau_cv


**Lecture :** une MAE/RMSE plus basse et un R² plus élevé indiquent généralement de meilleures prédictions. La RMSE pénalise davantage les grosses erreurs. La colonne RMSE moyenne aide à choisir le candidat à optimiser.


## 2. Optimiser le Random Forest

On teste peu de combinaisons pour garder la recherche facile à comprendre. `n_estimators` est le nombre d'arbres. `max_depth` limite leur profondeur. `min_samples_leaf` fixe le nombre minimal d'exemples dans une feuille. Les valeurs testées couvrent des arbres de tailles différentes et des feuilles plus ou moins simples.


In [ ]:
# Le prétraitement reste dans le pipeline : il est réappris dans chaque fold.
rf_pipeline = Pipeline([
    ("preparation", preprocessor),
    ("modele", RandomForestRegressor(random_state=42, n_jobs=-1)),
])

parametres = {
    "modele__n_estimators": [100, 200],
    "modele__max_depth": [None, 20],
    "modele__min_samples_leaf": [1, 2],
}

recherche = GridSearchCV(
    rf_pipeline,
    param_grid=parametres,
    scoring="neg_root_mean_squared_error",
    cv=folds,
    n_jobs=-1,
)
recherche.fit(X_train, y_train)

print("Meilleurs paramètres :", recherche.best_params_)
print("RMSE moyenne en validation :", round(-recherche.best_score_, 2))


## 3. Comparaison finale sur le jeu de test

Le jeu de test n'a pas servi à choisir les paramètres. On l'utilise maintenant une seule fois pour comparer le Random Forest initial et la version optimisée.


In [ ]:
# La fonction évite de répéter les mêmes calculs pour chaque modèle.
def mesurer_modele(nom, modele, X_eval, y_eval):
    predictions = modele.predict(X_eval)
    return {
        "Modèle": nom,
        "MAE": mean_absolute_error(y_eval, predictions),
        "RMSE": np.sqrt(mean_squared_error(y_eval, predictions)),
        "R²": r2_score(y_eval, predictions),
    }

rf_initial = fitted_models["Random Forest"]
rf_optimise = recherche.best_estimator_

comparaison_finale = pd.DataFrame([
    mesurer_modele("Random Forest initial", rf_initial, X_test, y_test),
    mesurer_modele("Random Forest optimisé", rf_optimise, X_test, y_test),
]).sort_values("RMSE")
comparaison_finale.round(2)


## 4. Examiner les erreurs du modèle retenu

On retient ici le Random Forest optimisé. Une erreur positive signifie que le modèle a surestimé le prix ; une erreur négative signifie qu'il l'a sous-estimé.


In [ ]:
modele_retenu = rf_optimise
prix_predits = modele_retenu.predict(X_test)

analyse = pd.DataFrame({
    "Prix réel": y_test,
    "Prix prédit": prix_predits,
}, index=y_test.index)
analyse["Erreur"] = analyse["Prix prédit"] - analyse["Prix réel"]
analyse["Erreur absolue"] = analyse["Erreur"].abs()

# Exemples pour lesquels le modèle s'est le plus trompé.
plus_grosses_erreurs = analyse.sort_values("Erreur absolue", ascending=False).head(10)
plus_grosses_erreurs.round(0)


In [ ]:
# Prix réels et prix estimés : les points proches de la diagonale sont les plus précis.
plt.figure(figsize=(7, 6))
plt.scatter(analyse["Prix réel"], analyse["Prix prédit"], alpha=0.6)
minimum = min(analyse["Prix réel"].min(), analyse["Prix prédit"].min())
maximum = max(analyse["Prix réel"].max(), analyse["Prix prédit"].max())
plt.plot([minimum, maximum], [minimum, maximum], color="red")
plt.xlabel("Prix réel")
plt.ylabel("Prix prédit")
plt.title("Prix réels et prix prédits")
plt.tight_layout()
plt.show()


In [ ]:
# Résidus = prix réel moins prix prédit. Une forme équilibrée autour de zéro est souhaitable.
residus = analyse["Prix réel"] - analyse["Prix prédit"]
plt.figure(figsize=(7, 5))
plt.scatter(analyse["Prix prédit"], residus, alpha=0.6)
plt.axhline(0, color="red")
plt.xlabel("Prix prédit")
plt.ylabel("Résidu (réel - prédit)")
plt.title("Résidus selon le prix prédit")
plt.tight_layout()
plt.show()

plt.figure(figsize=(7, 5))
plt.hist(analyse["Erreur"], bins=30, edgecolor="white")
plt.axvline(0, color="red")
plt.xlabel("Erreur (prédit - réel)")
plt.ylabel("Nombre de logements")
plt.title("Distribution des erreurs")
plt.tight_layout()
plt.show()


### Comment interpréter les erreurs ?

- Des points loin de la diagonale représentent des estimations éloignées du prix réel.
- Des résidus répartis autour de zéro indiquent peu de biais global. Une forme ou une tendance visible peut révéler que le modèle se trompe davantage pour certains niveaux de prix.
- Les dix logements listés plus haut sont ceux qui méritent une vérification. Les caractéristiques disponibles ne suffisent pas toujours à expliquer leur prix ; il faut comparer leurs surfaces, qualité, quartier et équipements avant de proposer une cause.


## 5. Variables importantes

Le Random Forest donne un score d'importance à chaque colonne après transformation. Ces scores montrent quelles colonnes le modèle a le plus utilisées ; ils ne prouvent pas qu'elles causent le prix.


In [ ]:
# Les noms incluent les catégories créées par l'encodage automatique.
noms_variables = modele_retenu.named_steps["preparation"].get_feature_names_out()
importances = modele_retenu.named_steps["modele"].feature_importances_

variables_importantes = pd.DataFrame({
    "Variable": noms_variables,
    "Importance": importances,
}).sort_values("Importance", ascending=False).head(15)

variables_importantes.sort_values("Importance").plot.barh(
    x="Variable", y="Importance", legend=False, figsize=(8, 6)
)
plt.title("15 variables les plus importantes")
plt.xlabel("Importance dans le modèle")
plt.tight_layout()
plt.show()


## Bilan du Jour 4

Consultez les tableaux pour noter le modèle le plus stable en validation croisée et vérifier si l'optimisation améliore réellement le score sur le jeu de test. Les graphiques et les exemples d'erreurs servent à comprendre les limites du modèle. Les tendances observées doivent ensuite être comparées à l'EDA (surface, qualité et quartier).
